<a href="https://colab.research.google.com/github/naman-0804/learning/blob/Langchain/vectorless_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
#!pip install pageindex

In [25]:
import google.generativeai as genai
from google.colab import userdata

# Securely fetch the API keys from Colab secrets
try:
    # Existing Gemini configuration
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Fetch the additional Page Index API Key
    PAGEINDEX_API_KEY = userdata.get('PAGEINDEX_API_KEY')

    print("Gemini API configured successfully.")
    print("PAGEINDEX_API_KEY successfully retrieved.")
except Exception as e:
    print(f"Error fetching keys: {e}")
    print("Ensure 'GEMINI_API_KEY' and 'PAGEINDEX_API_KEY' are set in the secrets tab (🔑).")

Gemini API configured successfully.
PAGEINDEX_API_KEY successfully retrieved.


In [26]:
from pageindex import PageIndexClient
PDF_PATH="./CitiIndia_FullTime_AppDev_JD.pdf"
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
result=pi_client.submit_document(PDF_PATH)
doc_id=result["doc_id"]
print(f"Document ID: {doc_id} ,  save this")

Document ID: pi-cmsy54sxw00e901p525kpl5uk ,  save this


In [27]:
#Poll untill processing is complete

In [28]:
import time

print("Building Tree Index")
while True:
  status_result = pi_client.get_document(doc_id)
  status = status_result.get("status")
  print(f" Status: {status}")

  if status == "completed":
    print("\n Tree index ready")
    break
  elif status == "failed":
    print("\n Processing failed")
    break
  time.sleep(5)

Building Tree Index
 Status: processing
 Status: processing
 Status: completed

 Tree index ready


In [29]:
import json
#inspecting the tree structure
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get("result",[])
print(f"Top-level sections:{len(pageindex_tree)}")
print("\n Raw Tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else{},indent=2))

Top-level sections:3

 Raw Tree (first node):
{
  "title": "Technology \u2013 Application Development, Full Time Analyst, India 2026",
  "node_id": "0000",
  "page_index": 1,
  "summary": "The Citi Technology Analyst Program is a 2-year rotational initiative in India designed to develop early-career technologists through agile software development, structured training, and mentorship. Participants gain exposure to diverse technical stacks\u2014including microservices, mobile, and web development\u2014while adhering to engineering excellence standards like CI/CD, containerization, and architectural best practices. The program emphasizes global collaboration, continuous professional growth, and the opportunity to contribute to impactful, real-world financial technology solutions.",
  "text": "# Technology \u2013 Application Development, Full Time Analyst, India 2026\n\nYou are the brains behind our work ...\n\nAt Citi, we do not just adapt to change \u2013 we drive it. Our Summer Technol

In [30]:
import json

# Fetch the full tree structure without truncation
full_tree_result = pi_client.get_tree(doc_id, node_summary=True)
full_tree = full_tree_result.get("result", [])

# Pretty print the entire tree
print(f"Full Document Tree for: {doc_id}\n")
print(json.dumps(full_tree, indent=2))

Full Document Tree for: pi-cmsy54sxw00e901p525kpl5uk

[
  {
    "title": "Technology \u2013 Application Development, Full Time Analyst, India 2026",
    "node_id": "0000",
    "page_index": 1,
    "summary": "The Citi Technology Analyst Program is a 2-year rotational initiative in India designed to develop early-career technologists through agile software development, structured training, and mentorship. Participants gain exposure to diverse technical stacks\u2014including microservices, mobile, and web development\u2014while adhering to engineering excellence standards like CI/CD, containerization, and architectural best practices. The program emphasizes global collaboration, continuous professional growth, and the opportunity to contribute to impactful, real-world financial technology solutions.",
    "text": "# Technology \u2013 Application Development, Full Time Analyst, India 2026\n\nYou are the brains behind our work ...\n\nAt Citi, we do not just adapt to change \u2013 we drive 

In [31]:
# Perform a search and use Gemini for the answer
query = "What are the graduation year and degree requirements for this program?"

# 1. Retrieve relevant context using the query method
# Note: PageIndex typically uses .query() for retrieval
try:
    search_results = pi_client.query(doc_id, query)
    context_text = search_results.get("context", "No context found.")
except AttributeError:
    # Fallback if the library uses search_nodes or similar
    search_results = pi_client.search_nodes(doc_id, query)
    context_text = " ".join([node.get('text', '') for node in search_results.get('result', [])])

# 2. Use Gemini to generate the answer based on that context
model = genai.GenerativeModel('gemini-1.5-flash')
prompt = f"""Based on the following document context, answer the user's query.

Query: {query}

Context:
{context_text}
"""

response = model.generate_content(prompt)

print(f"--- Search Results for: '{query}' ---\n")
print(response.text)

print("\n--- Retrieved Context ---")
print(context_text)

AttributeError: 'PageIndexClient' object has no attribute 'search'